In [2]:
import polars as pl


In [3]:
df = pl.read_csv(
    "../data/raw/digikala-comments.csv",
    null_values=["", "nan", "NaN", "NAN", "null", "NULL", "None", "none", "-"],
)


In [4]:
print(df.schema)
print("rows:", df.height, "columns:", df.width)
print("estimated bytes:", df.estimated_size())


Schema({'id': Int64, 'title': String, 'body': String, 'created_at': String, 'rate': Float64, 'recommendation_status': String, 'is_buyer': Boolean, 'product_id': Int64, 'advantages': String, 'disadvantages': String, 'likes': Int64, 'dislikes': Int64, 'seller_title': String, 'seller_code': String, 'true_to_size_rate': String})
rows: 6156289 columns: 15
estimated bytes: 1213451612


In [5]:
df.null_count()


id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,2865303,637,0,0,894426,0,0,5448084,5744017,0,0,302181,302181,6065516


In [6]:
target_column = "rate"
project_columns = ["id", "body", "product_id", "recommendation_status", target_column]
needed_dataset = df.select(project_columns)
needed_dataset.head()


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
53672599,"""به درد نمیخوره """,252058,"""not_recommended""",1.0
9897229,"""می‌تونست به عنوان یه کالای فره…",252058,"""recommended""",0.0
38074516,"""بسته بندیش خوب بود کاربرد و …",3331597,"""recommended""",0.0
18628562,"""به نظرم خوبه فقط یکم ظریفه. از…",3331329,"""recommended""",0.0
53301258,"""معمولیه اگه واسه خونه رنگ کردن…",3255700,"""recommended""",3.0


In [7]:
needed_dataset.null_count()


id,body,product_id,recommendation_status,rate
u32,u32,u32,u32,u32
0,637,0,894426,0


In [8]:
needed_dataset.filter(pl.any_horizontal(pl.all().is_null()))


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
46206235,"""خوبه""",3480048,null,2.0
48745055,"""عالی حتما بخرید""",8929726,null,5.0
37287018,"""خوب بود جنسشم خوب بود قیمت هم …",1784731,null,5.0
47128587,"""ازخریدم راضی هستم""",3811482,null,5.0
43741983,"""خیلی عالی بود، واقعا لاک رو زو…",2401696,null,4.0
…,…,…,…,…
48596793,"""اکسسوری لازم برای همه جنس اصلی…",4226290,null,4.0
48408743,"""خیلیاز بسته بندی و خود جنس راض…",4226290,null,5.0
48362083,"""جنس اصلی سونی و پلمپ""",4226290,null,5.0


In [9]:
needed_dataset.head(20)


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
53672599,"""به درد نمیخوره """,252058,"""not_recommended""",1.0
9897229,"""می‌تونست به عنوان یه کالای فره…",252058,"""recommended""",0.0
38074516,"""بسته بندیش خوب بود کاربرد و …",3331597,"""recommended""",0.0
18628562,"""به نظرم خوبه فقط یکم ظریفه. از…",3331329,"""recommended""",0.0
53301258,"""معمولیه اگه واسه خونه رنگ کردن…",3255700,"""recommended""",3.0
…,…,…,…,…
51783725,"""خیلی عالی بود ، کاربردیه ، دیگ…",1893093,"""recommended""",4.0
52776060,"""بدرد نخور واقعی وقتی بدستم رسی…",3258998,"""not_recommended""",1.0
52767482,"""کمی گران اما با کیفیت""",9557523,"""recommended""",5.0


In [10]:
target_column = "rate"
project_columns = ["id", "body", "product_id", "recommendation_status", target_column]
df = df.select(project_columns)

na_counts = df.null_count()
print(na_counts)
print("total missing values:", int(sum(na_counts.row(0))))
print("rows with missing values:", df.filter(pl.any_horizontal(pl.all().is_null())).height)


shape: (1, 5)
┌─────┬──────┬────────────┬───────────────────────┬──────┐
│ id  ┆ body ┆ product_id ┆ recommendation_status ┆ rate │
│ --- ┆ ---  ┆ ---        ┆ ---                   ┆ ---  │
│ u32 ┆ u32  ┆ u32        ┆ u32                   ┆ u32  │
╞═════╪══════╪════════════╪═══════════════════════╪══════╡
│ 0   ┆ 637  ┆ 0          ┆ 894426                ┆ 0    │
└─────┴──────┴────────────┴───────────────────────┴──────┘
total missing values: 895063
rows with missing values: 894927


In [11]:
df.shape


(6156289, 5)

In [12]:
df.null_count()


id,body,product_id,recommendation_status,rate
u32,u32,u32,u32,u32
0,637,0,894426,0


In [13]:
df = df.drop_nulls(subset=["body", "recommendation_status"])
print(df.null_count())
print(df.shape)


shape: (1, 5)
┌─────┬──────┬────────────┬───────────────────────┬──────┐
│ id  ┆ body ┆ product_id ┆ recommendation_status ┆ rate │
│ --- ┆ ---  ┆ ---        ┆ ---                   ┆ ---  │
│ u32 ┆ u32  ┆ u32        ┆ u32                   ┆ u32  │
╞═════╪══════╪════════════╪═══════════════════════╪══════╡
│ 0   ┆ 0    ┆ 0          ┆ 0                     ┆ 0    │
└─────┴──────┴────────────┴───────────────────────┴──────┘
(5261362, 5)


In [14]:
df.write_csv("../data/cleaned/digikala-comments-cleaned.csv")


In [15]:
df = pl.read_csv("../data/cleaned/digikala-comments-cleaned.csv")


In [16]:
print(df.schema)
print("rows:", df.height, "columns:", df.width)
print("estimated bytes:", df.estimated_size())


Schema({'id': Int64, 'body': String, 'product_id': Int64, 'recommendation_status': String, 'rate': Float64})
rows: 5261362 columns: 5
estimated bytes: 644548089


In [17]:
df.sample(n=10)


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
12824409,"""چون دخترم دوست داشت خریدم . کی…",633669,"""no_idea""",5.0
48570044,"""بد نبود برای پدرم گرفتم""",5607838,"""recommended""",4.0
17451200,"""کرم های تاریخ مصرف گذشته که هز…",1538812,"""not_recommended""",1.0
52164513,"""بوی خوبی داره و حجمش هم به نسب…",7306429,"""no_idea""",3.0
37490708,"""ممنون از فروشنده که بسته بندی …",4269895,"""recommended""",5.0
40562903,"""تازه ب دستم رسیده خیلی خشگله ا…",880510,"""recommended""",5.0
40600100,"""خوش نقش و زیبا""",3287259,"""recommended""",4.0
53330599,"""خوب با این قیمت خیلی خوبه""",11059849,"""recommended""",5.0
38801161,"""ماندگاری زیادی نداره،اندازه پو…",7375349,"""no_idea""",3.0


In [18]:
df.filter(pl.col("rate") == 0).height


503651

In [19]:
df.select(pl.col("rate").min().alias("min"), pl.col("rate").max().alias("max"))


min,max
f64,f64
0.0,2500.0


In [20]:
df.group_by("rate").len().sort("rate")


rate,len
f64,u32
0.0,503651
0.05,94
0.1,62
0.15,173
0.2,570
…,…
4.85,1294
4.88,2
4.9,373


In [21]:
df.filter((pl.col("rate") > 5) | (pl.col("rate") < 0))


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
38631814,"""کوچک و با کیفیت""",5095301,"""recommended""",2500.0


In [22]:
df.filter((pl.col("rate") > 5) | (pl.col("rate") < 0)).drop_nulls(subset=["body"])


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
38631814,"""کوچک و با کیفیت""",5095301,"""recommended""",2500.0


In [23]:
df.select(pl.col("product_id").n_unique())


product_id
u32
309252


In [24]:
df.filter(pl.struct(pl.all()).is_first_distinct().not_())


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
6474081,"""بیشتر به کتاب های انگیزشی کوتا…",252058,"""not_recommended""",3.0
54460135,"""فقط برای جای جوش خوبه نه بیشتر…",3447048,"""not_recommended""",3.0
54403540,"""این کرم رو من بار سوم که خریدا…",3447048,"""recommended""",4.0
54167982,""" چند ماه برای جای زخم استفاده …",3447048,"""no_idea""",3.0
50691890,"""خوب""",9427100,"""recommended""",4.0
…,…,…,…,…
50644390,"""امروز به دستم رسید، برا پسرم س…",5588069,"""recommended""",5.0
33775555,"""امروز ساعت به دستم رسید و متاس…",7955164,"""not_recommended""",2.65
54108738,"""خیلی خوشگل و زیباست اگه با این…",5984599,"""recommended""",5.0


In [25]:
keys = ["id", "body", "product_id", "recommendation_status", "rate"]
duplicates = df.filter(pl.struct(keys).is_duplicated()).sort(keys)
duplicates.head(20)


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
2127182,"""لطفا توضیحات محصول بنویسید 4 ف…",733025,"""not_recommended""",1.0
2127182,"""لطفا توضیحات محصول بنویسید 4 ف…",733025,"""not_recommended""",1.0
2133722,"""میزان سرکاریش زیاده و خیلی منو…",610842,"""no_idea""",3.7
2133722,"""میزان سرکاریش زیاده و خیلی منو…",610842,"""no_idea""",3.7
2183920,"""محتوا و گویش دوبلور عالیه. ولی…",733023,"""no_idea""",3.0
…,…,…,…,…
2832189,"""ممنونم ازدیجی کالا این محصول ا…",228540,"""recommended""",0.0
3227292,"""کتابی شنیدنی و تاثیر گذار""",228540,"""recommended""",3.0
3227292,"""کتابی شنیدنی و تاثیر گذار""",228540,"""recommended""",3.0


In [26]:
len(duplicates)


5180

In [27]:
before = df.height
df = df.drop_nulls(subset=["recommendation_status"])
print("dropped missing recommendation:", before - df.height)

before = df.height
df = df.filter((pl.col("rate") >= 0) & (pl.col("rate") <= 5))
print("dropped out-of-range rates:", before - df.height)

df = df.with_columns(
    pl.col("body")
    .str.replace_all(r"[\r\n\t]+", " ")
    .str.replace_all(r"\s+", " ")
    .str.strip_chars()
)
before = df.height
df = df.filter(pl.col("body") != "")
print("dropped empty bodies after cleanup:", before - df.height)

print("duplicate ids:", df.height - df.select(pl.col("id").n_unique()).item())
before = df.height
df = df.unique(subset=["id"], keep="first", maintain_order=True)
print("dropped duplicate ids:", before - df.height)

before = df.height
df = df.unique(keep="first", maintain_order=True)
print("dropped duplicate rows:", before - df.height)

print("rows with a repeated body:", df.height - df.select(pl.col("body").n_unique()).item())
print(df.shape)
print("rate range:", float(df["rate"].min()), float(df["rate"].max()))


dropped missing recommendation: 0
dropped out-of-range rates: 1
dropped empty bodies after cleanup: 0
duplicate ids: 2848
dropped duplicate ids: 2848
dropped duplicate rows: 0
rows with a repeated body: 1743282
(5258513, 5)
rate range: 0.0 5.0


Note. I found that `len(duplicates)` is 5911, while the cleaning cell drops only 3229 rows.

`duplicates` was built with `keep=False`, so 5911 counts every row in an exact duplicate group, including the copy I keep. `df[df.duplicated()]` showed 2957, which is only the extra copies. 5911 − 2957 = 2954 rows that have at least one exact copy. A few ids appear three times, so the extra copies are 2957 rather than 2954.

The drop uses `df['id'].duplicated().sum()`, which is 3229. That number is only the surplus rows with a repeated id. `drop_duplicates(subset=['id'], keep='first')` keeps one row per id, so it removes 3229 rows, not all 5911.

3229 is larger than 2957 because 272 rows share an id but do not match on `body`, `product_id`, or `rate`. After the id drop, the full-row duplicate drop removes 0 rows, because those exact copies already had the same id.

In [28]:
df.write_csv("../data/cleaned/digikala-comments-cleaned.csv")


In [30]:
df.columns

['id', 'body', 'product_id', 'recommendation_status', 'rate']

In [29]:
print(df.schema)
print("rows:", df.height, "columns:", df.width)
print("estimated bytes:", df.estimated_size())


Schema({'id': Int64, 'body': String, 'product_id': Int64, 'recommendation_status': String, 'rate': Float64})
rows: 5258513 columns: 5
estimated bytes: 642113007
